# 노트북 4. 컨텍스트 윈도우와 토큰

> Phase 2 — 제어: 모델 동작을 원하는 대로 다루기

토큰은 LLM의 화폐입니다.
비용도 토큰, 성능 한계도 토큰, 응답 품질도 토큰에 달렸습니다.

**학습 목표**
- 토큰의 실체(서브워드 단위)를 이해하고 직접 측정할 수 있다
- 한국어와 영어의 토큰 효율 차이를 설명할 수 있다
- 컨텍스트 윈도우의 개념과 한계를 이해한다
- 멀티턴 대화에서의 토큰 누적과 비용 구조를 계산할 수 있다
- Long Context vs RAG의 트레이드오프를 설명할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 토큰 실체 + count_tokens + 컨텍스트 윈도우 + 비용 + Long Context vs RAG | 읽고 실행 |
| Part 2 — 실습 | 토큰 측정 + 비용 계산 + 윈도우 초과 재현 | 직접 작성 |
| Part 3 — 챌린지 | 20턴 대화의 비용 추정 시스템 설계 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai

import os
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-2.5-flash"
print("환경 설정 완료")

---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 토큰의 실체

LLM은 텍스트를 단어 단위가 아니라 **토큰(token)** 단위로 처리합니다.
토큰은 **서브워드(subword)** 단위입니다.
하나의 단어가 여러 토큰으로 쪼개질 수도 있고, 짧은 단어는 하나의 토큰이 될 수도 있습니다.

예시:
- `"hello"` → 1토큰
- `"unhappiness"` → `["un", "happiness"]` → 2토큰
- `"인공지능"` → 여러 토큰 (한국어는 더 많은 토큰 소비)

> 핵심: 토큰은 LLM이 텍스트를 이해하는 최소 단위입니다.
> 비용, 속도, 컨텍스트 한계 모두 토큰 수로 결정됩니다.

### count_tokens() API

Gemini는 `client.models.count_tokens()` API로 텍스트의 토큰 수를 직접 측정할 수 있습니다.
실제 API 호출을 하지 않으므로 비용이 들지 않습니다.

아래 코드는 간단한 문장의 토큰 수를 측정합니다.

In [ ]:
# count_tokens()로 토큰 수 측정
result = client.models.count_tokens(
    model=MODEL,
    contents="Hello, world!",
)

print(f"'Hello, world!' → {result.total_tokens} 토큰")

여러 문장의 토큰 수를 비교해봅시다.
단어 수와 토큰 수는 반드시 일치하지 않습니다.

In [ ]:
# 다양한 영어 텍스트의 토큰 수
english_texts = [
    "Hello",
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming the world.",
    "supercalifragilisticexpialidocious",  # 매우 긴 단어
]

print(f"{'텍스트':<55} {'단어수':>5} {'토큰수':>5}")
print("-" * 70)
for text in english_texts:
    token_count = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    word_count = len(text.split())
    print(f"{text:<55} {word_count:>5} {token_count:>5}")

### 한국어 vs 영어 토큰 효율

한국어는 영어보다 **토큰 효율이 낮습니다**.
같은 의미를 전달하는 데 더 많은 토큰을 소비합니다.
이것은 한국어 사용자에게 비용과 컨텍스트 윈도우 측면에서 불리합니다.

아래 코드는 같은 의미의 한국어/영어 문장의 토큰 수를 비교합니다.

In [ ]:
# 동일 의미 한국어/영어 토큰 비교
pairs = [
    ("Hello", "안녕하세요"),
    ("What is artificial intelligence?", "인공지능이란 무엇인가요?"),
    ("The capital of South Korea is Seoul.", "대한민국의 수도는 서울입니다."),
    ("Python is a popular programming language.", "파이썬은 인기 있는 프로그래밍 언어입니다."),
    ("Machine learning models learn patterns from data.", "머신러닝 모델은 데이터에서 패턴을 학습합니다."),
]

print(f"{'영어':<50} {'EN':>4} {'한국어':<35} {'KO':>4} {'배율':>5}")
print("-" * 100)
for en, ko in pairs:
    en_tokens = client.models.count_tokens(model=MODEL, contents=en).total_tokens
    ko_tokens = client.models.count_tokens(model=MODEL, contents=ko).total_tokens
    ratio = ko_tokens / en_tokens if en_tokens > 0 else 0
    print(f"{en:<50} {en_tokens:>4} {ko:<35} {ko_tokens:>4} {ratio:>5.1f}x")

> 핵심: 한국어는 영어 대비 약 1.5~2배의 토큰을 사용합니다.
> 같은 대화를 한국어로 하면 비용이 더 많이 들고, 컨텍스트 윈도우도 더 빨리 차게 됩니다.

### 토큰 수 어림 규칙

API 호출 없이 토큰 수를 대략적으로 추정하는 경험 규칙이 있습니다.
정확하지는 않지만, 비용이나 컨텍스트 사용량을 빠르게 어림할 때 유용합니다.

| 언어 | 어림 규칙 | 근거 |
|------|----------|------|
| 영어 | 1토큰 ≈ 4글자 (또는 0.75단어) | 라틴 알파벳 기반 서브워드 |
| 한국어 | 1토큰 ≈ 1~2글자 | 유니코드 다바이트 문자, 조합형 |
| 코드 | 단어 수와 비슷 | 식별자, 기호가 개별 토큰 |

아래 코드는 이 어림 규칙이 실제 토큰 수와 얼마나 차이 나는지 확인합니다.

In [ ]:
# 어림 규칙 vs 실제 토큰 수 비교
test_cases = [
    ("영어", "The quick brown fox jumps over the lazy dog.", lambda t: len(t) / 4),
    ("영어", "Artificial intelligence is the future of technology.", lambda t: len(t) / 4),
    ("한국어", "인공지능은 기술의 미래입니다.", lambda t: len(t) / 1.5),
    ("한국어", "오늘 날씨가 매우 좋습니다.", lambda t: len(t) / 1.5),
    ("코드", "def fibonacci(n):\n    if n <= 1: return n", lambda t: len(t.split())),
]

print(f"{'유형':<6} {'텍스트':<50} {'어림':>5} {'실제':>5} {'오차':>6}")
print("-" * 80)
for lang, text, estimator in test_cases:
    estimated = estimator(text)
    actual = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    error = ((estimated - actual) / actual) * 100
    print(f"{lang:<6} {text:<50} {estimated:>5.0f} {actual:>5} {error:>+5.0f}%")

### 다양한 유형의 텍스트 토큰화

텍스트 유형에 따라 토큰 효율이 달라집니다.
코드, 숫자, 특수 문자는 일반 텍스트와 다른 토큰화 패턴을 보입니다.

아래 코드는 다양한 유형의 텍스트가 어떻게 토큰화되는지 비교합니다.

In [ ]:
# 다양한 유형의 텍스트 토큰화
varied_texts = {
    "일반 영어": "The weather is nice today.",
    "일반 한국어": "오늘 날씨가 좋습니다.",
    "파이썬 코드": "def fibonacci(n):\n    if n <= 1: return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "JSON 데이터": '{"name": "Kim", "age": 30, "city": "Seoul"}',
    "숫자 나열": "1234567890 9876543210 1111111111",
    "URL": "https://www.example.com/api/v2/users?page=1&limit=10",
    "긴 반복": "ha " * 50,  # "ha ha ha ..." 50번
}

print(f"{'유형':<12} {'글자수':>6} {'토큰수':>6} {'글자/토큰':>8}")
print("-" * 40)
for label, text in varied_texts.items():
    tokens = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    chars = len(text)
    ratio = chars / tokens if tokens > 0 else 0
    print(f"{label:<12} {chars:>6} {tokens:>6} {ratio:>8.1f}")

## 1.2 count_tokens() 심화

`count_tokens()`는 단순 텍스트뿐 아니라 실제 API 호출과 동일한 형태의 입력을 측정할 수 있습니다.
`system_instruction`을 포함한 전체 토큰 수도 계산 가능합니다.

### system_instruction 포함 토큰 수

실제 API 호출에서는 `system_instruction`도 토큰으로 소비됩니다.
system prompt가 길면 매 호출마다 그만큼의 토큰이 추가됩니다.

아래 코드는 system_instruction의 토큰 수를 확인합니다.

In [ ]:
# system_instruction 포함 토큰 수 측정
system_prompt = "당신은 친절한 한국어 도우미입니다. 간결하게 답변하세요."
user_message = "파이썬이란 무엇인가요?"

# system_instruction 없이
tokens_without = client.models.count_tokens(
    model=MODEL,
    contents=user_message,
).total_tokens

# system_instruction 포함
tokens_with = client.models.count_tokens(
    model=MODEL,
    contents=user_message,
    config={"system_instruction": system_prompt},
).total_tokens

# system_instruction 자체의 토큰 수
tokens_system_only = client.models.count_tokens(
    model=MODEL,
    contents=system_prompt,
).total_tokens

print(f"user만: {tokens_without} 토큰")
print(f"system + user: {tokens_with} 토큰")
print(f"system 자체: {tokens_system_only} 토큰")
print(f"차이: {tokens_with - tokens_without} 토큰 (system 기여분)")

### 멀티턴 대화의 토큰 측정

멀티턴 대화에서는 매 호출마다 전체 이력이 전송됩니다.
`count_tokens()`로 실제 전송될 토큰 수를 사전에 확인할 수 있습니다.

아래 코드는 대화가 쌓이면서 토큰 수가 어떻게 증가하는지 보여줍니다.

In [ ]:
from google.genai.types import Content, Part

# 가상 멀티턴 대화 이력
conversation = [
    Content(role="user", parts=[Part(text="안녕하세요, 저는 김철수입니다.")]),
    Content(role="model", parts=[Part(text="안녕하세요, 김철수님! 무엇을 도와드릴까요?")]),
    Content(role="user", parts=[Part(text="파이썬 공부를 시작하려고 합니다.")]),
    Content(role="model", parts=[Part(text="좋은 선택입니다! 파이썬은 초보자에게 추천하는 언어입니다.")]),
    Content(role="user", parts=[Part(text="어디서부터 시작하면 좋을까요?")]),
]

# 턴별 누적 토큰 수 확인
print(f"{'턴':>3} {'누적 메시지':>10} {'토큰 수':>8}")
print("-" * 25)
for i in range(1, len(conversation) + 1):
    partial = conversation[:i]
    tokens = client.models.count_tokens(model=MODEL, contents=partial).total_tokens
    print(f"{i:>3} {len(partial):>10}개 {tokens:>8}")

### Content/Part 구조체와 토큰

지금까지 `contents`에 문자열을 직접 전달했지만, 실제로는 `Content` 객체와 `Part` 객체로 구조화할 수 있습니다.
문자열 전달과 구조체 전달의 토큰 수가 동일한지 확인해봅시다.

아래 코드는 같은 텍스트를 다른 방식으로 전달했을 때 토큰 수를 비교합니다.

In [ ]:
# 같은 텍스트를 다른 형태로 전달했을 때 토큰 수 비교
text = "오늘 날씨가 어떤가요?"

# 방법 1: 문자열 직접 전달
t1 = client.models.count_tokens(model=MODEL, contents=text).total_tokens

# 방법 2: Content 객체로 전달
t2 = client.models.count_tokens(
    model=MODEL,
    contents=Content(role="user", parts=[Part(text=text)]),
).total_tokens

# 방법 3: 리스트로 감싸서 전달
t3 = client.models.count_tokens(
    model=MODEL,
    contents=[Content(role="user", parts=[Part(text=text)])],
).total_tokens

print(f"문자열 직접: {t1} 토큰")
print(f"Content 객체: {t2} 토큰")
print(f"리스트 감싸기: {t3} 토큰")
print(f"\n→ 전달 방식에 관계없이 동일한 텍스트는 동일한 토큰 수입니다.")

## 1.3 컨텍스트 윈도우

**컨텍스트 윈도우(Context Window)**는 모델이 한 번에 처리할 수 있는 최대 토큰 수입니다.
입력 토큰과 출력 토큰의 합이 이 한계를 넘을 수 없습니다.

```
입력 토큰 + 출력 토큰 <= 컨텍스트 윈도우
```

### Gemini 모델별 컨텍스트 윈도우

| 모델 | 컨텍스트 윈도우 | 최대 출력 토큰 |
|------|----------------|---------------|
| gemini-2.5-flash | 1,048,576 (약 100만) | 65,536 |
| gemini-2.5-pro | 1,048,576 (약 100만) | 65,536 |
| gemini-2.0-flash | 1,048,576 (약 100만) | 8,192 |

100만 토큰은 약 70만 단어, 책 약 10권 분량에 해당합니다.

> 핵심: 컨텍스트 윈도우가 크다고 해서 항상 좋은 것은 아닙니다.
> 긴 컨텍스트에서는 모델의 주의력이 분산되는 **Lost in the Middle** 현상이 발생할 수 있습니다.
> 중간에 위치한 정보는 앞이나 뒤에 있는 정보보다 덜 잘 인식됩니다.

### 컨텍스트 사용률 계산

실제 호출에서 컨텍스트 윈도우를 얼마나 사용했는지 확인해봅시다.

아래 코드는 호출 후 사용된 토큰과 윈도우 한계를 비교합니다.

In [ ]:
# 컨텍스트 사용률 확인
CONTEXT_WINDOW = 1_048_576  # gemini-2.5-flash

response = client.models.generate_content(
    model=MODEL,
    contents="대한민국의 역사를 간략히 설명해주세요.",
)

usage = response.usage_metadata
total_used = usage.prompt_token_count + usage.candidates_token_count
usage_pct = (total_used / CONTEXT_WINDOW) * 100

print(f"입력 토큰: {usage.prompt_token_count:,}")
print(f"출력 토큰: {usage.candidates_token_count:,}")
print(f"합계: {total_used:,}")
print(f"컨텍스트 윈도우: {CONTEXT_WINDOW:,}")
print(f"사용률: {usage_pct:.4f}%")

### usage_metadata 상세 분석

API 응답의 `usage_metadata`에는 여러 필드가 포함되어 있습니다.
어떤 정보를 제공하는지 전체를 확인해봅시다.

아래 코드는 `usage_metadata`의 모든 필드를 출력합니다.

In [ ]:
# usage_metadata 전체 필드 확인
response = client.models.generate_content(
    model=MODEL,
    contents="파이썬의 장점을 3가지 알려주세요.",
)

usage = response.usage_metadata
print("=== usage_metadata 필드 ===")
print(f"prompt_token_count:     {usage.prompt_token_count:>8,}  # 입력 토큰 수")
print(f"candidates_token_count: {usage.candidates_token_count:>8,}  # 출력 토큰 수")
print(f"total_token_count:      {usage.total_token_count:>8,}  # 입력 + 출력")

# thinking_token_count는 모델이 thinking을 지원할 때만 존재
thinking_tokens = getattr(usage, 'thoughts_token_count', None)
if thinking_tokens is not None:
    print(f"thoughts_token_count:   {thinking_tokens:>8,}  # 추론(thinking) 토큰")
else:
    print(f"thoughts_token_count:   해당 없음 (thinking 비활성화 상태)")

print(f"\n검증: prompt({usage.prompt_token_count}) + candidates({usage.candidates_token_count}) = {usage.prompt_token_count + usage.candidates_token_count}")
print(f"       total_token_count = {usage.total_token_count}")

단일 호출에서는 윈도우의 극히 일부만 사용합니다.
하지만 멀티턴 대화가 누적되면 상황이 달라집니다.

### 컨텍스트 윈도우 초과 시

입력이 컨텍스트 윈도우를 초과하면 API가 에러를 반환합니다.
이 에러를 직접 확인해보는 것이 중요합니다.

아래 코드는 매우 긴 텍스트를 전송하여 윈도우 한계를 테스트합니다.
(Gemini 2.5 Flash의 윈도우가 100만 토큰으로 매우 크기 때문에,
먼저 긴 텍스트의 토큰 수를 측정하고, 윈도우 대비 사용률을 확인합니다.)

In [ ]:
# 매우 긴 텍스트 생성 후 토큰 수 확인
long_text = "이것은 매우 긴 텍스트입니다. " * 5000  # 약 5000번 반복

token_count = client.models.count_tokens(
    model=MODEL,
    contents=long_text,
).total_tokens

print(f"반복 텍스트 길이: {len(long_text):,} 글자")
print(f"토큰 수: {token_count:,}")
print(f"윈도우 대비: {(token_count / CONTEXT_WINDOW) * 100:.2f}%")

In [ ]:
# 토큰이 매우 많은 텍스트로 호출 시도
# 윈도우 내라면 정상 동작하지만, 초과 시 에러 발생
try:
    response = client.models.generate_content(
        model=MODEL,
        contents=long_text,
        config={"max_output_tokens": 10},  # 출력은 최소화
    )
    print(f"성공! 응답 토큰: {response.usage_metadata.candidates_token_count}")
except Exception as e:
    print(f"에러 발생: {type(e).__name__}")
    print(f"메시지: {e}")

## 1.4 비용 구조

Gemini API 비용은 **토큰 단위**로 과금됩니다.
입력 토큰과 출력 토큰의 단가가 다르며, 출력 토큰이 더 비쌉니다.

### Gemini 2.5 Flash 가격 (2025년 기준)

| 구분 | 단가 (100만 토큰당) |
|------|-------------------|
| 입력 토큰 (≤200K) | $0.15 |
| 입력 토큰 (>200K) | $0.30 |
| 출력 토큰 (≤200K) | $0.60 |
| 출력 토큰 (>200K) | $1.20 |

> 주의: 가격은 변동될 수 있습니다.
> 최신 가격은 Google AI Studio 또는 Google Cloud 문서에서 확인하세요.
> 무료 티어도 있으며, 분당 요청 수 제한 등의 조건이 있습니다.

### 비용 계산 함수

토큰 수를 기반으로 비용을 계산하는 유틸리티 함수를 만들어봅시다.

아래 코드는 비용 계산 함수와 실제 호출 결과에 적용하는 예시를 보여줍니다.

In [ ]:
def calculate_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float = 0.15,
    output_price_per_million: float = 0.60,
) -> dict:
    """토큰 수로 비용을 계산 (USD)"""
    input_cost = (input_tokens / 1_000_000) * input_price_per_million
    output_cost = (output_tokens / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "input_cost_usd": input_cost,
        "output_cost_usd": output_cost,
        "total_cost_usd": total_cost,
        "total_cost_krw": total_cost * 1400,  # 대략적 환율
    }


# 실제 호출에 적용
response = client.models.generate_content(
    model=MODEL,
    contents="파이썬의 장점을 5가지 알려주세요.",
)

usage = response.usage_metadata
cost = calculate_cost(usage.prompt_token_count, usage.candidates_token_count)

print(f"입력: {cost['input_tokens']:,} 토큰 → ${cost['input_cost_usd']:.6f}")
print(f"출력: {cost['output_tokens']:,} 토큰 → ${cost['output_cost_usd']:.6f}")
print(f"합계: ${cost['total_cost_usd']:.6f} (약 {cost['total_cost_krw']:.4f}원)")

단일 호출은 매우 저렴합니다.
하지만 멀티턴 대화에서 토큰이 누적되면 비용이 기하급수적으로 증가합니다.

### 모델별 비용 비교

같은 호출이라도 사용하는 모델에 따라 비용이 크게 달라집니다.
Gemini 모델 간 가격 차이를 비교해봅시다.

| 모델 | 입력 ($/1M) | 출력 ($/1M) | Flash 대비 |
|------|-----------|-----------|----------|
| gemini-2.5-flash | $0.15 | $0.60 | 1x |
| gemini-2.5-pro | $1.25 | $10.00 | ~12x |
| gemini-2.0-flash | $0.10 | $0.40 | ~0.7x |

아래 코드는 동일한 호출을 각 모델 가격으로 계산합니다.

In [ ]:
# 모델별 비용 비교 시뮬레이션
# 가정: 입력 500토큰, 출력 300토큰
INPUT_TOKENS = 500
OUTPUT_TOKENS = 300

models_pricing = {
    "gemini-2.5-flash": (0.15, 0.60),
    "gemini-2.5-pro":   (1.25, 10.00),
    "gemini-2.0-flash": (0.10, 0.40),
}

print(f"{'모델':<22} {'입력비용($)':>12} {'출력비용($)':>12} {'합계($)':>12} {'합계(원)':>10}")
print("-" * 72)
base_cost = None
for model_name, (in_price, out_price) in models_pricing.items():
    cost = calculate_cost(INPUT_TOKENS, OUTPUT_TOKENS, in_price, out_price)
    if base_cost is None:
        base_cost = cost["total_cost_usd"]
    ratio = cost["total_cost_usd"] / base_cost
    print(
        f"{model_name:<22} "
        f"{cost['input_cost_usd']:>12.8f} "
        f"{cost['output_cost_usd']:>12.8f} "
        f"{cost['total_cost_usd']:>12.8f} "
        f"{cost['total_cost_krw']:>9.5f}"
    )
print(f"\n→ gemini-2.5-pro는 Flash 대비 약 {models_pricing['gemini-2.5-pro'][1] / models_pricing['gemini-2.5-flash'][1]:.0f}배 비쌉니다.")
print(f"  성능이 필요한 경우에만 사용하고, 일반 용도에는 Flash가 비용 효율적입니다.")

### 멀티턴 토큰 누적과 비용

멀티턴 대화에서 각 턴마다 이전 전체 이력이 입력에 포함됩니다.
10턴 대화의 비용은 단순히 "1턴 비용 x 10"이 아닙니다.
누적 합산이므로 훨씬 더 많습니다.

아래 코드는 멀티턴 대화에서의 토큰 누적과 비용을 시뮬레이션합니다.

In [ ]:
# 멀티턴 비용 시뮬레이션
# 가정: 매 턴 user 30토큰, AI 100토큰, system 20토큰
SYSTEM_TOKENS = 20
USER_TOKENS_PER_TURN = 30
AI_TOKENS_PER_TURN = 100

print(f"{'턴':>3} {'입력 토큰':>10} {'출력 토큰':>10} {'누적 비용($)':>12} {'누적 비용(원)':>12}")
print("-" * 55)

total_cost_usd = 0
for turn in range(1, 21):  # 20턴
    # 입력 = system + 이전 모든 턴의 (user + AI) + 현재 user
    input_tokens = SYSTEM_TOKENS + (turn - 1) * (USER_TOKENS_PER_TURN + AI_TOKENS_PER_TURN) + USER_TOKENS_PER_TURN
    output_tokens = AI_TOKENS_PER_TURN

    cost = calculate_cost(input_tokens, output_tokens)
    total_cost_usd += cost["total_cost_usd"]
    total_cost_krw = total_cost_usd * 1400

    if turn <= 5 or turn % 5 == 0:  # 앞 5턴 + 5턴마다 출력
        print(f"{turn:>3} {input_tokens:>10,} {output_tokens:>10,} {total_cost_usd:>12.6f} {total_cost_krw:>12.4f}")

> 핵심: 멀티턴 대화의 비용은 턴 수에 따라 **이차(quadratic)**적으로 증가합니다.
> 1~5턴에서는 무시할 수준이지만, 20턴이 넘으면 비용이 체감될 수 있습니다.
> 이것이 노트북 7(컨텍스트 매니지먼트)에서 이력 관리 전략을 배워야 하는 이유입니다.

### Thinking 토큰과 비용

Gemini 2.5 모델은 **Thinking(추론)** 기능을 지원합니다.
Thinking이 활성화되면 모델이 답변 전에 내부적으로 "생각"하는 과정을 거치며, 이 과정에서도 토큰이 소비됩니다.

Thinking 관련 토큰은 3종류입니다:

| 토큰 종류 | 설명 | 과금 |
|----------|------|------|
| Input tokens | 사용자 입력 + system prompt | 입력 단가 |
| Output tokens | 실제 응답 텍스트 | 출력 단가 |
| Thinking tokens | 내부 추론 과정 | 출력 단가와 동일 |

> 주의: thinking_budget를 높게 설정하면 추론 토큰이 크게 증가할 수 있습니다.
> 단순한 질문에는 thinking을 비활성화하는 것이 비용 효율적입니다.
> Thinking에 대해서는 노트북 10에서 자세히 다룹니다.

아래 코드는 Thinking 활성화/비활성화 시 토큰 사용량 차이를 비교합니다.

In [ ]:
# Thinking 활성화/비활성화 토큰 비교
question = "7 + 13은 얼마인가요?"

# thinking 비활성화 (budget=0)
resp_no_think = client.models.generate_content(
    model=MODEL,
    contents=question,
    config={"thinking_config": {"thinking_budget": 0}},
)

# thinking 활성화 (기본값)
resp_think = client.models.generate_content(
    model=MODEL,
    contents=question,
    config={"thinking_config": {"thinking_budget": 1024}},
)

u1 = resp_no_think.usage_metadata
u2 = resp_think.usage_metadata

# thinking 토큰 확인
think_tokens = getattr(u2, 'thoughts_token_count', 0) or 0

print(f"{'항목':<20} {'Thinking OFF':>15} {'Thinking ON':>15}")
print("-" * 55)
print(f"{'입력 토큰':<20} {u1.prompt_token_count:>15,} {u2.prompt_token_count:>15,}")
print(f"{'출력 토큰':<20} {u1.candidates_token_count:>15,} {u2.candidates_token_count:>15,}")
print(f"{'Thinking 토큰':<20} {'0':>15} {think_tokens:>15,}")
print(f"{'전체 토큰':<20} {u1.total_token_count:>15,} {u2.total_token_count:>15,}")

### 응답에서 토큰 정보 추출

실제 API 호출 후 `usage_metadata`에서 정확한 토큰 사용량을 확인할 수 있습니다.

아래 코드는 실제 멀티턴 호출에서 토큰 누적을 관찰합니다.

In [ ]:
# 실제 멀티턴 호출에서 토큰 누적 관찰
contents = []
system = "한 문장으로 답변하세요."

questions = [
    "파이썬이란?",
    "자바와의 차이는?",
    "어떤 걸 먼저 배울까요?",
    "웹 개발에는?",
    "데이터 분석에는?",
]

cumulative_cost = 0
print(f"{'턴':>3} {'입력':>8} {'출력':>8} {'합계':>8} {'누적비용($)':>12}")
print("-" * 45)

for i, q in enumerate(questions, 1):
    contents.append(Content(role="user", parts=[Part(text=q)]))

    resp = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config={
            "system_instruction": system,
            "thinking_config": {"thinking_budget": 0},
        },
    )

    contents.append(Content(role="model", parts=[Part(text=resp.text)]))

    u = resp.usage_metadata
    cost = calculate_cost(u.prompt_token_count, u.candidates_token_count)
    cumulative_cost += cost["total_cost_usd"]

    print(f"{i:>3} {u.prompt_token_count:>8,} {u.candidates_token_count:>8,} {u.total_token_count:>8,} {cumulative_cost:>12.6f}")

## 1.5 Long Context vs RAG

Gemini의 100만 토큰 윈도우는 매우 큽니다.
문서를 통째로 넣을 수도 있습니다.
하지만 **모든 상황에서 그렇게 하는 것이 최선은 아닙니다**.

문서를 활용하는 두 가지 전략:

1. **Long Context**: 문서 전체를 프롬프트에 넣기
2. **RAG**: 관련 부분만 검색하여 프롬프트에 넣기

| 기준 | Long Context | RAG |
|------|-------------|-----|
| 구현 복잡도 | 낮음 (그냥 넣으면 됨) | 높음 (임베딩 + 벡터DB + 검색) |
| 비용 (호출당) | 높음 (전체 문서 토큰) | 낮음 (검색된 청크만) |
| 정확도 | 문서 전체를 볼 수 있음 | 검색 품질에 의존 |
| 속도 | 느림 (긴 입력 처리) | 빠름 (짧은 입력) |
| Lost in the Middle | 발생 가능 | 덜 발생 (짧은 컨텍스트) |
| 문서 업데이트 | 즉시 반영 | 재인덱싱 필요 |

> 핵심: 문서가 짧고(10페이지 이하), 호출 빈도가 낮으면 Long Context가 간편합니다.
> 문서가 길거나 호출이 빈번하면 RAG가 비용 효율적입니다.
> RAG에 대해서는 노트북 12(임베딩)와 13(RAG)에서 자세히 다룹니다.

아래 코드는 같은 질문에 대해 Long Context와 짧은 Context의 비용 차이를 시뮬레이션합니다.

In [ ]:
# Long Context vs Short Context(RAG) 비용 비교

# 시나리오: 50페이지 문서(약 50,000 토큰)에서 질문에 답하기
# 100번 질문한다고 가정

DOC_TOKENS = 50_000  # 전체 문서
CHUNK_TOKENS = 2_000  # RAG로 검색된 청크 (상위 4개 x 500토큰)
QUESTION_TOKENS = 30
ANSWER_TOKENS = 200
NUM_CALLS = 100

# Long Context: 매번 전체 문서 + 질문
long_cost = 0
for _ in range(NUM_CALLS):
    c = calculate_cost(DOC_TOKENS + QUESTION_TOKENS, ANSWER_TOKENS)
    long_cost += c["total_cost_usd"]

# RAG: 매번 검색된 청크 + 질문
rag_cost = 0
for _ in range(NUM_CALLS):
    c = calculate_cost(CHUNK_TOKENS + QUESTION_TOKENS, ANSWER_TOKENS)
    rag_cost += c["total_cost_usd"]

print(f"=== 100회 질문 비용 비교 ===")
print(f"Long Context: ${long_cost:.4f} (약 {long_cost * 1400:.1f}원)")
print(f"RAG:          ${rag_cost:.4f} (약 {rag_cost * 1400:.1f}원)")
print(f"절감률: {(1 - rag_cost / long_cost) * 100:.1f}%")

### 비용 증가 추이 시각화

멀티턴 대화에서 비용이 어떤 곡선으로 증가하는지 텍스트 차트로 확인해봅시다.
선형 증가(1턴 비용 x N)와 실제 이차 증가의 차이를 비교합니다.

아래 코드는 20턴까지의 누적 비용을 텍스트 바 차트로 시각화합니다.

In [ ]:
# 비용 증가 추이: 선형 가정 vs 실제(이차)
SYSTEM = 20
USER = 30
AI = 100

# 1턴 비용 (기준)
first_turn_cost = calculate_cost(SYSTEM + USER, AI)["total_cost_usd"]

linear_cumulative = 0
actual_cumulative = 0

print(f"{'턴':>3} {'선형(가정)':>12} {'실제(이차)':>12} {'차이':>8}")
print("-" * 40)

for turn in range(1, 21):
    # 선형 가정: 매 턴 동일한 비용
    linear_cumulative += first_turn_cost

    # 실제: 이전 이력이 누적
    input_tokens = SYSTEM + (turn - 1) * (USER + AI) + USER
    actual_cumulative += calculate_cost(input_tokens, AI)["total_cost_usd"]

    if turn in [1, 5, 10, 15, 20]:
        diff = actual_cumulative / linear_cumulative if linear_cumulative > 0 else 1
        print(f"{turn:>3} ${linear_cumulative:>11.6f} ${actual_cumulative:>11.6f} {diff:>7.1f}x")

print(f"\n→ 20턴 시점에서 실제 비용은 선형 가정의 약 {diff:.1f}배입니다.")

# 텍스트 바 차트
print(f"\n=== 턴별 입력 토큰 증가 (막대) ===")
max_bar = 50
max_tokens = SYSTEM + 19 * (USER + AI) + USER
for turn in range(1, 21):
    input_tokens = SYSTEM + (turn - 1) * (USER + AI) + USER
    bar_len = int((input_tokens / max_tokens) * max_bar)
    print(f"{turn:>2} | {'#' * bar_len:<{max_bar}} {input_tokens:>6}")

---

### 생각해보기

1. 한국어가 영어보다 토큰 효율이 낮다면, 한국어 서비스를 운영할 때 비용을 줄이는 방법은 무엇이 있을까요?
2. 컨텍스트 윈도우가 100만 토큰이라면 사실상 무제한이라고 볼 수 있을까요? 여전히 윈도우를 의식해야 하는 이유는 무엇일까요?
3. Long Context가 간편하다면 왜 RAG가 여전히 필요할까요? 비용 외에 다른 이유가 있을까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 4-1: 다양한 텍스트의 토큰 수 측정

아래 텍스트들의 토큰 수를 `count_tokens()`로 측정하고 표로 출력하세요.

**요구사항**
1. `client.models.count_tokens()`를 사용
2. 아래 5가지 텍스트를 측정:
   - `"Hello, world!"`
   - `"안녕하세요, 세계!"`
   - `"def hello(): print('hi')"`
   - `"12345 67890 11111"`
   - `"https://www.google.com/search?q=gemini"`
3. 각 텍스트의 글자 수와 토큰 수를 출력

In [ ]:
# TODO: 5가지 텍스트의 토큰 수를 측정하세요
texts = [
    "Hello, world!",
    "안녕하세요, 세계!",
    "def hello(): print('hi')",
    "12345 67890 11111",
    "https://www.google.com/search?q=gemini",
]

# ---------- 정답 ----------
# print(f"{'텍스트':<45} {'글자수':>6} {'토큰수':>6}")
# print("-" * 60)
# for text in texts:
#     tokens = client.models.count_tokens(model=MODEL, contents=text).total_tokens
#     print(f"{text:<45} {len(text):>6} {tokens:>6}")

print("TODO를 완성해주세요")

### 실습 4-2: 한국어/영어 토큰 효율 비교

동일한 의미의 한국어/영어 문장 쌍을 3개 이상 작성하고 토큰 수를 비교하세요.

**요구사항**
1. 한국어/영어 쌍을 최소 3개 작성 (직접 작성)
2. 각 쌍의 토큰 수를 측정
3. 한국어가 영어 대비 몇 배의 토큰을 사용하는지 계산
4. 평균 배율을 출력

In [ ]:
# TODO: 한국어/영어 쌍을 작성하고 토큰 효율을 비교하세요
pairs = [
    # ("English sentence", "한국어 문장"),
]  # 최소 3개의 쌍을 작성하세요

# ---------- 정답 ----------
# pairs = [
#     ("I like coffee.", "저는 커피를 좋아합니다."),
#     ("The weather is cold today.", "오늘 날씨가 춥습니다."),
#     ("Please recommend a good restaurant.", "좋은 식당을 추천해주세요."),
# ]

if pairs:
    ratios = []
    print(f"{'영어':<40} {'EN':>4} {'한국어':<25} {'KO':>4} {'배율':>6}")
    print("-" * 85)
    for en, ko in pairs:
        en_t = client.models.count_tokens(model=MODEL, contents=en).total_tokens
        ko_t = client.models.count_tokens(model=MODEL, contents=ko).total_tokens
        ratio = ko_t / en_t if en_t > 0 else 0
        ratios.append(ratio)
        print(f"{en:<40} {en_t:>4} {ko:<25} {ko_t:>4} {ratio:>6.2f}x")
    print(f"\n평균 배율: {sum(ratios)/len(ratios):.2f}x")
else:
    print("TODO를 완성해주세요")

### 실습 4-3: system_instruction의 토큰 비용 확인

긴 system prompt와 짧은 system prompt가 토큰 수에 어떤 영향을 미치는지 확인하세요.

**요구사항**
1. 짧은 system prompt: "간결하게 답변하세요."
2. 긴 system prompt: 5줄 이상의 상세한 규칙 포함 (직접 작성)
3. 동일한 질문("파이썬이란?")에 대해 각각 `count_tokens()` 실행
4. 토큰 수 차이를 출력

In [ ]:
question = "파이썬이란?"

# TODO 1: 짧은 system prompt
short_system = ""  # 이 문자열을 작성하세요

# TODO 2: 긴 system prompt (5줄 이상)
long_system = ""  # 이 문자열을 작성하세요

# ---------- 정답 ----------
# short_system = "간결하게 답변하세요."
# long_system = (
#     "당신은 10년 경력의 소프트웨어 엔지니어입니다.\n"
#     "초보자가 이해할 수 있도록 쉽게 설명하세요.\n"
#     "반드시 예시 코드를 포함하세요.\n"
#     "각 개념을 설명할 때 비유를 사용하세요.\n"
#     "답변은 500자 이내로 작성하세요.\n"
#     "마지막에 핵심 요약을 한 줄로 추가하세요."
# )

# 검증
if short_system and long_system:
    t_short = client.models.count_tokens(
        model=MODEL, contents=question,
        config={"system_instruction": short_system},
    ).total_tokens

    t_long = client.models.count_tokens(
        model=MODEL, contents=question,
        config={"system_instruction": long_system},
    ).total_tokens

    print(f"짧은 system: {t_short} 토큰")
    print(f"긴 system:   {t_long} 토큰")
    print(f"차이: {t_long - t_short} 토큰 (매 호출마다 추가됨)")
else:
    print("TODO를 완성해주세요")

### 실습 4-4: 멀티턴 토큰 누적 시뮬레이션 + 비용 계산

10턴 멀티턴 대화를 시뮬레이션하고, 턴별 입력 토큰 수와 누적 비용을 계산하세요.

**요구사항**
1. 가정값 설정:
   - system: 30 토큰
   - 매 턴 user: 25 토큰
   - 매 턴 AI: 80 토큰
2. `calculate_cost()` 함수 사용 (이론에서 정의한 것)
3. 10턴까지 각 턴의 입력 토큰, 출력 토큰, 누적 비용을 출력
4. 마지막에 전체 누적 비용과 원화 환산 출력

In [ ]:
# TODO: 멀티턴 비용 시뮬레이션
SYSTEM = 30
USER_PER_TURN = 25
AI_PER_TURN = 80

# ---------- 정답 ----------
# total_accumulated = 0
# print(f"{'턴':>3} {'입력토큰':>10} {'출력토큰':>10} {'턴비용($)':>12} {'누적비용($)':>12}")
# print("-" * 50)
# for turn in range(1, 11):
#     input_tokens = SYSTEM + (turn - 1) * (USER_PER_TURN + AI_PER_TURN) + USER_PER_TURN
#     output_tokens = AI_PER_TURN
#     cost = calculate_cost(input_tokens, output_tokens)
#     total_accumulated += cost["total_cost_usd"]
#     print(f"{turn:>3} {input_tokens:>10,} {output_tokens:>10,} {cost['total_cost_usd']:>12.7f} {total_accumulated:>12.7f}")
#
# print(f"\n총 비용: ${total_accumulated:.6f} (약 {total_accumulated * 1400:.4f}원)")

print("TODO를 완성해주세요")

### 실습 4-5: 실제 멀티턴 호출에서 토큰 추적

실제 API를 호출하면서 턴별 토큰 사용량과 비용을 추적하세요.

**요구사항**
1. system_instruction: "한 문장으로 답변하세요."
2. 5개 질문으로 멀티턴 대화 실행
3. 각 턴의 `usage_metadata`에서 토큰 수 추출
4. `calculate_cost()`로 비용 계산
5. 턴별 입력/출력 토큰, 누적 비용을 표로 출력

In [ ]:
from google.genai.types import Content, Part

system = "한 문장으로 답변하세요."
questions = [
    "인공지능이란?",
    "머신러닝과의 차이는?",
    "딥러닝은?",
    "실생활 활용 예시는?",
    "앞으로의 전망은?",
]

# TODO: 멀티턴 대화 실행 + 토큰/비용 추적
contents = []

# ---------- 정답 ----------
# accumulated_cost = 0
# print(f"{'턴':>3} {'입력':>8} {'출력':>8} {'턴비용($)':>12} {'누적비용($)':>12}")
# print("-" * 50)
# for i, q in enumerate(questions, 1):
#     contents.append(Content(role="user", parts=[Part(text=q)]))
#     resp = client.models.generate_content(
#         model=MODEL,
#         contents=contents,
#         config={
#             "system_instruction": system,
#             "thinking_config": {"thinking_budget": 0},
#         },
#     )
#     contents.append(Content(role="model", parts=[Part(text=resp.text)]))
#
#     u = resp.usage_metadata
#     cost = calculate_cost(u.prompt_token_count, u.candidates_token_count)
#     accumulated_cost += cost["total_cost_usd"]
#     print(f"{i:>3} {u.prompt_token_count:>8,} {u.candidates_token_count:>8,} {cost['total_cost_usd']:>12.7f} {accumulated_cost:>12.7f}")
#
# print(f"\n총 비용: ${accumulated_cost:.6f} (약 {accumulated_cost * 1400:.4f}원)")

if len(contents) == 0:
    print("TODO를 완성해주세요")

### 실습 4-6: Long Context vs RAG 비용 비교 계산

문서 크기별로 Long Context와 RAG의 비용 차이를 계산하세요.

**요구사항**
1. 문서 크기: 1,000 / 10,000 / 50,000 / 100,000 토큰
2. RAG 청크 크기: 고정 2,000 토큰
3. 질문 토큰: 30, 응답 토큰: 200
4. 100회 질문 시 각각의 총 비용 계산
5. 비용 차이와 절감률을 표로 출력

In [ ]:
# TODO: 문서 크기별 Long Context vs RAG 비용 비교
doc_sizes = [1_000, 10_000, 50_000, 100_000]
RAG_CHUNK = 2_000
Q_TOKENS = 30
A_TOKENS = 200
NUM_QUESTIONS = 100

# ---------- 정답 ----------
# print(f"{'문서 크기':>10} {'Long($)':>10} {'RAG($)':>10} {'절감률':>8}")
# print("-" * 42)
# for doc_size in doc_sizes:
#     long_total = sum(
#         calculate_cost(doc_size + Q_TOKENS, A_TOKENS)["total_cost_usd"]
#         for _ in range(NUM_QUESTIONS)
#     )
#     rag_total = sum(
#         calculate_cost(RAG_CHUNK + Q_TOKENS, A_TOKENS)["total_cost_usd"]
#         for _ in range(NUM_QUESTIONS)
#     )
#     saving = (1 - rag_total / long_total) * 100 if long_total > 0 else 0
#     print(f"{doc_size:>10,} {long_total:>10.4f} {rag_total:>10.4f} {saving:>7.1f}%")

print("TODO를 완성해주세요")

### 실습 4-7: 월간 비용 예산 계산기

월간 예산이 정해져 있을 때, 몇 건의 대화를 처리할 수 있는지 계산하세요.

**요구사항**
1. 월간 예산: $10 (약 14,000원)
2. 1건의 대화: 평균 10턴, system 30토큰, user 40토큰/턴, AI 150토큰/턴
3. 1건 대화의 총 비용을 계산 (10턴 누적)
4. 월간 예산으로 처리 가능한 대화 수를 출력
5. gemini-2.5-flash와 gemini-2.5-pro 두 모델로 각각 계산

In [ ]:
# TODO: 월간 비용 예산 계산기
MONTHLY_BUDGET_USD = 10
TURNS_PER_CONV = 10
SYS_TOKENS = 30
USER_TOKENS = 40
AI_TOKENS = 150

# ---------- 정답 ----------
# model_prices = {
#     "gemini-2.5-flash": (0.15, 0.60),
#     "gemini-2.5-pro": (1.25, 10.00),
# }
#
# for model_name, (in_price, out_price) in model_prices.items():
#     conv_cost = 0
#     for turn in range(1, TURNS_PER_CONV + 1):
#         input_t = SYS_TOKENS + (turn - 1) * (USER_TOKENS + AI_TOKENS) + USER_TOKENS
#         cost = calculate_cost(input_t, AI_TOKENS, in_price, out_price)
#         conv_cost += cost["total_cost_usd"]
#
#     max_convs = int(MONTHLY_BUDGET_USD / conv_cost)
#     print(f"=== {model_name} ===")
#     print(f"1건 대화 비용: ${conv_cost:.6f} (약 {conv_cost * 1400:.4f}원)")
#     print(f"월간 처리 가능: {max_convs:,}건")
#     print(f"일간 처리 가능: {max_convs // 30:,}건")
#     print()

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 4-5에서 실제 토큰 사용량이 시뮬레이션(실습 4-4)과 어떻게 달랐나요? 차이가 나는 이유는 무엇일까요?
2. system prompt가 길면 매 호출마다 그만큼의 토큰이 추가됩니다. system prompt를 짧게 유지하면서도 효과적으로 만드는 방법은 무엇일까요?
3. 실습 4-7에서 Flash와 Pro의 월간 처리량 차이가 얼마나 컸나요? 어떤 상황에서 Pro의 추가 비용을 감수할 가치가 있을까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 4-1: 20턴 대화의 비용 추정 시스템 설계 (난이도: ★★☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

20턴 멀티턴 대화의 비용을 사전에 추정하는 시스템을 설계하세요.

**요구사항**
- 입력: system prompt, 예상 턴 수, 예상 사용자 메시지 길이, 예상 AI 응답 길이
- 출력: 턴별 예상 입력 토큰, 누적 비용 그래프(텍스트), 총 비용 요약
- 실제 `count_tokens()`를 사용하여 system prompt의 토큰 수를 정확히 측정할 것
- 비용 절감을 위한 제안 사항을 1-2가지 포함할 것

**평가 기준**
- 입력값을 변경했을 때 결과가 합리적으로 변하는가?
- 비용 절감 제안이 구체적인가?

**힌트**
- 실제 토큰 수는 메시지 길이로 근사할 수 있습니다 (한국어: 글자수 x 1.5~2)
- 비용 절감: system prompt 최적화, 응답 길이 제한, 이력 트리밍 등

In [ ]:
# 1단계: 비용 추정 함수 설계
# 여기에 코드를 작성하세요

In [ ]:
# 2단계: 20턴 시뮬레이션 실행
# 여기에 코드를 작성하세요

In [ ]:
# 3단계: 비용 절감 제안
# 여기에 분석과 제안을 작성하세요

---

### 생각해보기

1. 비용 추정 시스템에서 가장 큰 불확실성은 무엇인가요? 실제 비용과 추정 비용의 오차를 줄이려면 어떻게 해야 할까요?
2. 만약 토큰 비용이 10배 비싼 모델(예: gemini-2.5-pro)을 사용해야 한다면, 비용 관리 전략이 어떻게 달라져야 할까요?
3. 비용 절감과 응답 품질은 트레이드오프 관계입니다. 비용을 줄이면서도 품질을 유지하는 가장 효과적인 방법은 무엇이라고 생각하나요?